In [1]:
%pip install fastapi uvicorn scikit-learn pandas numpy mlflow jupyter

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

In [5]:
mlflow.set_tracking_uri("http://localhost:5000")

data = load_diabetes(scaled=False)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
experiment_name = "diabetes_experiment"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

with mlflow.start_run(experiment_id=experiment_id, run_name="random_forest_with_scaler"):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("train_r2", train_score)
    mlflow.log_metric("test_r2", test_score)
    
    mlflow.sklearn.log_model(pipeline, "model")
    
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    mlflow.register_model(model_uri, "diabetes_model")
    
    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print("Модель зарегистрирована как 'diabetes_model'")

2026/03/20 17:52:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 17:52:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'diabetes_model'.
2026/03/20 17:52:26 WARNING mlflow.tracking._model_registry.fluent: Run with id a8ef8a3a361b4a228c224f40b3754f5f has no artifacts at artifact path 'model', registering model based on models:/m-f76adb83fc2b4180a2f80a0e7b5807ae instead
2026/03/20 17:52:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes_model, version 1
Created version '1' of model 'diabetes_

Run ID: a8ef8a3a361b4a228c224f40b3754f5f
Модель зарегистрирована как 'diabetes_model'
🏃 View run random_forest_with_scaler at: http://localhost:5000/#/experiments/1/runs/a8ef8a3a361b4a228c224f40b3754f5f
🧪 View experiment at: http://localhost:5000/#/experiments/1
